# Air Quality Forecasting — Data Cleaning & Feature Engineering

This notebook prepares the raw Beijing PM2.5 hourly time series for modelling. We build a datetime index, engineer calendar / lag / rolling features, one-hot encode wind direction, drop rows with a missing target, and save the result to `data/airquality_features.csv` for notebook 03.

## 1. Imports & Load Data

We import our project utilities (which house the `create_features` function) and load the raw dataset, building the `Datetime` index from the year/month/day/hour columns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '.')
from utils import load_data, create_features

sns.set_style('whitegrid')
%matplotlib inline

df = load_data('data/PRSA_data.csv')
print('Raw shape:', df.shape)
df.head()

## 2. Handle Missing / Invalid Target

The target `pm2.5` is missing for a sizeable fraction of hours (sensor downtime). We **cannot** impute the target without inventing ground truth, so those rows will be dropped. We first sort the series ascending and deduplicate timestamps to guarantee a clean time order before any shifting.

In [ ]:
df = df.sort_values('Datetime').reset_index(drop=True)

n_dupes = df['Datetime'].duplicated().sum()
print('Duplicate timestamps:', n_dupes)
df = df.drop_duplicates(subset='Datetime').reset_index(drop=True)

print('Missing pm2.5:', df['pm2.5'].isna().sum(), f"({df['pm2.5'].isna().mean()*100:.1f}%)")
print('Missing in weather columns:')
print(df[['DEWP','TEMP','PRES','Iws','Is','Ir']].isna().sum())

## 3. Calendar Feature Engineering

Calendar features capture the deterministic, repeating seasonality in air quality. Month drives the heating-season cycle, hour-of-day captures the daily rhythm, and a weekend flag captures any weekday/weekend difference in traffic and industrial activity.

In [ ]:
df['hour']       = df['Datetime'].dt.hour
df['dayofweek']  = df['Datetime'].dt.dayofweek   # 0=Mon, 6=Sun
df['month']      = df['Datetime'].dt.month
df['quarter']    = df['Datetime'].dt.quarter
df['year']       = df['Datetime'].dt.year
df['dayofyear']  = df['Datetime'].dt.dayofyear
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

df[['Datetime','hour','dayofweek','month','quarter','year','dayofyear','is_weekend']].head()

## 4. Lag & Rolling-Mean Feature Engineering

Lag features capture autocorrelation: `lag_1` is the pollution one hour ago and `lag_24` is the reading at the same hour yesterday. Rolling means (`roll_24`, `roll_168`) smooth recent history over a day and a week. Every lag/rolling column is **shifted** so it only uses information available strictly before the current hour — this prevents data leakage from the target into the features.

In [ ]:
df['lag_1']  = df['pm2.5'].shift(1)
df['lag_24'] = df['pm2.5'].shift(24)
df['roll_24']  = df['pm2.5'].shift(1).rolling(window=24).mean()
df['roll_168'] = df['pm2.5'].shift(1).rolling(window=168).mean()

df[['pm2.5','lag_1','lag_24','roll_24','roll_168']].head(10)

## 5. One-Hot Encode Wind Direction

`cbwd` is the only categorical column. We one-hot encode it with `drop_first=True` to avoid the dummy trap, producing binary indicator columns for the wind-direction categories (one category becomes the reference).

In [ ]:
df = pd.get_dummies(df, columns=['cbwd'], prefix='wind', drop_first=True)
wind_cols = [c for c in df.columns if c.startswith('wind_')]
df[wind_cols] = df[wind_cols].astype(int)
print('Wind indicator columns:', wind_cols)
df[wind_cols].head()

## 6. Drop NaN Rows — Final Shape

We now drop every row with a missing target or a NaN from the lag/rolling warmup (the first 168 rows can't have a full weekly window). What remains is a fully numeric, leakage-free feature matrix.

In [ ]:
before = len(df)
df = df.dropna(subset=['pm2.5','lag_1','lag_24','roll_24','roll_168']).reset_index(drop=True)
after = len(df)
print(f'Dropped {before - after:,} rows (missing target / lag warmup)')
print(f'Final shape: {df.shape}')
print('Any remaining NaNs:', int(df.isna().sum().sum()))

## 7. Lag Feature Relationship with Target

Scatter plots of `lag_1` and `roll_24` against `pm2.5` (sampled for speed) confirm the strong persistence in air quality: the pollution one hour ago and the recent 24-hour average are excellent predictors of the current reading.

In [ ]:
sample = df.sample(5000, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(sample['lag_1'], sample['pm2.5'], alpha=0.3, s=8, color='steelblue')
axes[0].set_xlabel('lag_1 (µg/m³)'); axes[0].set_ylabel('pm2.5 (µg/m³)')
axes[0].set_title(f"lag_1 vs pm2.5 (r={df[['lag_1','pm2.5']].corr().iloc[0,1]:.3f})")
axes[1].scatter(sample['roll_24'], sample['pm2.5'], alpha=0.3, s=8, color='seagreen')
axes[1].set_xlabel('roll_24 (µg/m³)'); axes[1].set_ylabel('pm2.5 (µg/m³)')
axes[1].set_title(f"roll_24 vs pm2.5 (r={df[['roll_24','pm2.5']].corr().iloc[0,1]:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
feature_cols = ['hour','dayofweek','month','quarter','year','dayofyear','is_weekend',
                'DEWP','TEMP','PRES','Iws','Is','Ir',
                'lag_1','lag_24','roll_24','roll_168'] + wind_cols
corr_t = df[feature_cols + ['pm2.5']].corr()['pm2.5'].drop('pm2.5').sort_values()

plt.figure(figsize=(9, 6))
corr_t.plot(kind='barh', color='steelblue')
plt.title('Feature Correlation with PM2.5'); plt.xlabel('Pearson r')
plt.tight_layout(); plt.show()

## 8. Save Cleaned Feature CSV

We save the fully engineered DataFrame to `data/airquality_features.csv`. Notebook 03 loads this file directly and trains the regressors on it.

In [ ]:
out_path = 'data/airquality_features.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df):,} rows × {df.shape[1]} cols to {out_path}')
df.head()